# 04 · Sentence-transformers: embeddings and cross-encoders

After this notebook you can embed text with a bi-encoder, search and cluster with those embeddings, rerank candidates with a cross-encoder, explain exactly why production retrieval uses both, and fine-tune an embedding model with `MultipleNegativesRankingLoss`.

**Time:** ~25 min · **Runs:** offline on CPU in about a minute · **Needs:** [02 · Tokenizers and models](02_tokenizers_models_and_generate.ipynb) (mean pooling); [embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb) for the maths

## Why this matters in interviews

- Every RAG design question includes "which embedding model, which similarity, and do you rerank?". This library is how most teams answer it with open models.
- The bi-encoder vs cross-encoder distinction is the standard follow-up: why the reranker is more accurate, and why it can't be the first stage.
- Details like "normalise, then dot product equals cosine", `max_seq_length` truncation and batch size decide whether an index is right and fast. Interviewers probe them because they cause real bugs.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve01` | What is cosine similarity, and what are the alternatives? |
| `ve02` | When would you use dot product instead of cosine? |
| `ve23` | What is normalisation and when must you do it? |
| `ve17` | What is a cross-encoder and why can it not be used for first-stage retrieval? |
| `ve25` | What is the difference between an embedding model and a reranker model? |
| `tl13` | What is a reranker and where does it sit? |
| `rg08` | What is reranking, and why can a reranker not fix bad recall? |
| `tl18` | How do you pick an embedding model? |
| `rg38` | How would you fine-tune an embedding model for your domain? |
| `in34` | What is the difference between an embedding model call and a generation call, operationally? |

In [ ]:
import time, tempfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import transformers, datasets
import sentence_transformers
from sentence_transformers import SentenceTransformer, CrossEncoder, util

transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
datasets.disable_progress_bars()
torch.manual_seed(0)
EMB, RERANK = "sentence-transformers/all-MiniLM-L6-v2", "cross-encoder/ms-marco-MiniLM-L6-v2"
print("sentence-transformers", sentence_transformers.__version__)

## 1. A bi-encoder: text in, one vector out

**In plain English:** an embedding model turns a sentence into a list of numbers (384 of them here), placed so that sentences with similar meaning end up close together. You embed every document once, store the vectors, and at query time you embed only the question and look for the nearest vectors.

A `SentenceTransformer` is a small pipeline of modules. Print it to see them:

In [ ]:
model = SentenceTransformer(EMB, device="cpu")
print(model)
print("embedding dimension:", model.get_embedding_dimension(), "| max_seq_length:", model.max_seq_length,
      "| similarity:", model.similarity_fn_name)

Transformer (BERT, 6 layers) → **mean pooling** over tokens → **L2 normalisation**. That is exactly the recipe [notebook 02](02_tokenizers_models_and_generate.ipynb) rebuilt by hand. `encode` takes a list of strings and returns a `numpy` array of shape `[n, 384]`.

In [ ]:
sentences = ["A man is eating food.", "A man is eating a piece of bread.", "The girl is carrying a baby.",
             "A man is riding a horse.", "Someone is having a meal.", "A woman is playing violin."]
emb = model.encode(sentences, normalize_embeddings=True)
print("shape:", emb.shape, emb.dtype, "| norms:", np.linalg.norm(emb, axis=1).round(4))

sim = model.similarity(emb, emb)        # cosine for this model (it reads similarity_fn_name)
print(np.round(sim.numpy(), 2))
assert np.allclose(emb @ emb.T, sim.numpy(), atol=1e-5)      # unit vectors: dot product == cosine

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(sim.numpy(), cmap="viridis", vmin=-0.2, vmax=1)
labels = [s[:26] for s in sentences]
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right"); ax.set_yticks(range(len(labels)), labels)
ax.set_title("Cosine similarity between sentence embeddings"); plt.colorbar(im, fraction=0.046)
plt.show()

The "eating" sentences form a bright block, including "Someone is having a meal", which shares no content words with them. That is what dense embeddings add over keyword search (`fo04`). Unrelated pairs sit near 0, not at −1: cosine values for this kind of model mostly live between about −0.1 and 1.

**Normalisation (`ve23`, `ve02`):** once the vectors are unit length, `dot = cosine`, and every vector database's fastest metric (inner product) gives you cosine for free. The trap is mixing normalised and unnormalised vectors in one index, or normalising the documents and not the queries. Use raw dot product only if the model was *trained* for it (some models put useful information in the vector length). Check the model card.

## 2. Semantic search

`util.semantic_search` does the brute-force version of what a vector index does: score each query against every document, then keep the top-k. For a few thousand documents that is fast enough and exact. Beyond that you use an ANN index such as FAISS HNSW ([vector search notebook](../09_rag/03_vector_search_faiss_hnsw.ipynb)).

In [ ]:
docs = [
    "Standard shipping is free on all orders over 50 euros.",
    "Standard shipping takes 3 to 7 business days to arrive.",
    "Express shipping costs 15 euros and arrives in 1 to 2 business days.",
    "Refunds are issued to the original payment method.",
    "Refunds take up to 10 business days to appear on your card statement.",
    "You can request a refund within 30 days of delivery.",
    "Passwords must be at least 12 characters long.",
    "You can change your password in the account settings page.",
    "If you forget your password, use the reset link on the sign-in page.",
    "The warranty lasts two years from the date of purchase.",
    "The warranty does not cover water damage or accidental drops.",
    "To make a warranty claim, email a photo of the receipt to support.",
    "Our Berlin store is open from 10am to 8pm on weekdays.",
    "The Berlin store is closed on Sundays and public holidays.",
    "Parking is available behind the Berlin store.",
    "Gift cards never expire and can be used online or in store.",
]
doc_emb = model.encode(docs, normalize_embeddings=True, convert_to_tensor=True)    # the "index", built once

query = "is dropping my phone covered by the warranty"
hits = util.semantic_search(model.encode(query, normalize_embeddings=True, convert_to_tensor=True), doc_emb, top_k=3)[0]
print("query:", query)
for h in hits:
    print(f"  {h['score']:.3f}  {docs[h['corpus_id']]}")
assert hits[0]["corpus_id"] == 10

The right answer ranks first even though "dropping my phone" appears nowhere in it: the model maps "dropping" close to "accidental drops". Notice how close the three warranty documents score, though. A bi-encoder is good at finding the right *topic* and weaker at telling which document in that topic actually *answers* the question. That is the reranker's job.

## 3. Clustering embeddings

Embeddings also work for unsupervised grouping: topic discovery, deduplication, and sampling a diverse eval set. KMeans on normalised vectors behaves like clustering by cosine. PCA squashes 384 dimensions to 2 for a picture.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

topics = {
    "cooking":   ["Knead the dough and let it rise for an hour.", "Simmer the tomato sauce with garlic and basil.",
                  "Preheat the oven before baking the cake.", "Season the soup with salt and pepper.",
                  "Chop the onions finely for the stir-fry.", "Whisk the eggs with sugar until fluffy."],
    "sport":     ["The striker scored twice in the second half.", "She finished the marathon in under three hours.",
                  "The tennis final went to a fifth set.", "Our team won the championship match.",
                  "The goalkeeper saved a penalty.", "He broke the world record in the 100 metres."],
    "computers": ["The laptop needs a new graphics driver.", "Install the latest operating system update.",
                  "The app crashes when the phone runs out of memory.", "Restart the router to fix the Wi-Fi.",
                  "Python code runs faster with vectorised numpy.", "The server's hard drive is almost full."],
    "finance":   ["The central bank raised interest rates.", "Stock prices fell after the earnings report.",
                  "Pay off your credit card balance each month.", "Inflation reduced household purchasing power.",
                  "She moved her savings into an index fund.", "The company reported record quarterly profits."],
}
texts = [s for group in topics.values() for s in group]
truth = np.repeat(np.arange(len(topics)), 6)
X = model.encode(texts, normalize_embeddings=True)

km = KMeans(n_clusters=4, n_init=10, random_state=1).fit(X)
ari = adjusted_rand_score(truth, km.labels_)
print(f"adjusted Rand index vs the true topics: {ari:.2f}  (1 = perfect, 0 = random)")
assert ari > 0.6

xy = PCA(n_components=2, random_state=0).fit_transform(X)
fig, ax = plt.subplots(figsize=(8, 4.5))
markers = ["o", "s", "^", "D"]
for t, name in enumerate(topics):
    idx = truth == t
    ax.scatter(xy[idx, 0], xy[idx, 1], c=[f"C{c}" for c in km.labels_[idx]], marker=markers[t], s=70, label=name,
               edgecolors="k")
ax.set_title("Sentence embeddings, PCA to 2-D (marker = true topic, colour = KMeans cluster)")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.legend(fontsize=8)
plt.show()

The four topics separate well, but not perfectly. A few sentences land in the "wrong" cluster, typically ones that share a surface feature with another topic (a phone that runs out of memory, a record that is a number). Two-dimensional PCA also hides structure that exists in the full 384 dimensions, so points that overlap in the picture can still be separable.

## 4. Speed: batch size matters

Embedding is a single forward pass with no generation loop, so it batches extremely well (`in34`). Encoding one sentence at a time wastes most of the hardware. `encode(batch_size=...)` controls this (the default is 32).

In [ ]:
corpus = [f"Ticket {i}: the {w} on my {p} stopped working after the update."
          for i, (w, p) in enumerate(zip(np.resize(["screen", "battery", "camera", "speaker"], 128),
                                         np.resize(["phone", "laptop", "tablet"], 128)))]
batch_sizes = [1, 4, 16, 64]
throughput = []
model.encode(corpus[:8])                                          # warm-up
for bs in batch_sizes:
    t0 = time.perf_counter()
    model.encode(corpus, batch_size=bs)
    throughput.append(len(corpus) / (time.perf_counter() - t0))
    print(f"batch_size={bs:3d}: {throughput[-1]:6.0f} sentences/s")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(batch_sizes, throughput, "o-")
ax.set_xscale("log", base=2); ax.set_xticks(batch_sizes, batch_sizes)
ax.set_xlabel("batch_size"); ax.set_ylabel("sentences / second")
ax.set_title(f"Encoding throughput on this CPU ({len(corpus)} sentences)")
plt.show()
assert max(throughput[1:]) > 1.2 * throughput[0]

Batching helps until the hardware is saturated. On a laptop CPU that happens at small batch sizes, so the curve flattens early. On a GPU it keeps climbing into the hundreds, and one sentence at a time wastes most of the card. For a corpus backfill, use big batches and `convert_to_numpy`; for a live query you encode one string, and that latency (a few milliseconds on a GPU) is rarely the bottleneck in RAG.

## 5. Max sequence length and silent truncation

This model reads at most **256 tokens** (`max_seq_length`). Anything after that is **silently dropped**: no error, no warning at `encode` time. That is a real bug in RAG systems that embed whole documents instead of chunks.

In [ ]:
filler = "This sentence is filler text about nothing in particular. " * 40
doc_a = "Our returns policy is simple. " + filler
doc_b = doc_a + "The warehouse is in Rotterdam."          # the key fact comes after token 256
n_tokens = len(model.tokenizer(doc_b)["input_ids"])
ea, eb = model.encode([doc_a, doc_b], normalize_embeddings=True)
print(f"doc_b is {n_tokens} tokens; max_seq_length = {model.max_seq_length}")
print(f"cosine(doc_a, doc_b) = {ea @ eb:.6f}  -> the Rotterdam sentence changed nothing")
assert np.allclose(ea, eb, atol=1e-6)
q = model.encode("where is the warehouse", normalize_embeddings=True)
print(f"a query about the warehouse scores {q @ eb:.3f} against doc_b; against the sentence alone: "
      f"{q @ model.encode('The warehouse is in Rotterdam.', normalize_embeddings=True):.3f}")

That is why you **chunk** before embedding ([chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb)): chunks should fit the model's window with room to spare, and one idea per chunk keeps the vector focused. Different models have very different limits (256, 512, 8 k tokens), so check before you pick one (`tl18`).

## 6. Cross-encoders: rerank the candidates

**In plain English:** a bi-encoder describes the question and each document *separately* and compares the summaries. A cross-encoder reads the question and one document *together*, so every word of the query can attend to every word of the document, and it outputs one relevance score. It is more accurate, but it needs one forward pass per (query, document) pair, and nothing can be precomputed.

`CrossEncoder.predict` scores pairs, and `CrossEncoder.rank` sorts documents for one query:

In [ ]:
reranker = CrossEncoder(RERANK, device="cpu")
pairs = [("how long does standard shipping take", docs[0]), ("how long does standard shipping take", docs[1])]
print("predict:", reranker.predict(pairs).round(2), "(raw logits: higher = more relevant; not probabilities)")

for r in reranker.rank("is the Berlin store open on Sunday", docs[12:15], return_documents=True):
    print(f"  {r['score']:7.2f}  {r['text']}")

The standard two-stage pattern (`tl13`): the **bi-encoder retrieves** top-k candidates from the whole corpus in milliseconds, then the **cross-encoder reranks** only those k. Let's measure both stages on 14 labelled questions:

In [ ]:
eval_set = [("how long does standard shipping take", 1), ("how much does express delivery cost", 2),
            ("how long until the refund shows on my card", 4), ("where will my refund be sent", 3),
            ("what is the minimum password length", 6), ("I forgot my password", 8),
            ("is dropping my phone covered by the warranty", 10), ("how do I claim the warranty", 11),
            ("is the Berlin store open on Sunday", 13), ("where can I park near the Berlin shop", 14),
            ("do gift cards expire", 15), ("what is the deadline to ask for my money back", 5),
            ("how long is the warranty", 9), ("can I get free shipping", 0)]
K = 5
bi_top1, rr_top1, recall_k, bi_ms, rr_ms = [], [], [], [], []
for q, gold in eval_set:
    t0 = time.perf_counter()
    cand = [h["corpus_id"] for h in util.semantic_search(
        model.encode(q, normalize_embeddings=True, convert_to_tensor=True), doc_emb, top_k=K)[0]]
    t1 = time.perf_counter()
    ranked = reranker.rank(q, [docs[c] for c in cand])
    t2 = time.perf_counter()
    rr_best = cand[ranked[0]["corpus_id"]]
    bi_top1.append(cand[0] == gold); rr_top1.append(rr_best == gold); recall_k.append(gold in cand)
    bi_ms.append((t1 - t0) * 1000); rr_ms.append((t2 - t1) * 1000)
    if cand[0] != gold or rr_best != gold:
        print(f"{q!r}\n   bi-encoder #1: {docs[cand[0]]}\n   reranked   #1: {docs[rr_best]}\n   correct      : {docs[gold]}")

print(f"\ntop-1 accuracy: bi-encoder {np.mean(bi_top1):.2f} -> + reranker {np.mean(rr_top1):.2f} "
      f"| recall@{K} (the ceiling for any reranker) {np.mean(recall_k):.2f}")
print(f"median latency per query: retrieve {np.median(bi_ms):.1f} ms, rerank {K} docs {np.median(rr_ms):.1f} ms")
assert np.mean(rr_top1) >= np.mean(bi_top1) and np.mean(rr_top1) <= np.mean(recall_k)

Read the printed cases:

- **Where reranking helps:** for "open on Sunday", the bi-encoder picks the weekday-hours document (same topic, lots of shared words). The cross-encoder reads both together and picks the document that actually answers "Sunday".
- **Where it doesn't:** for "where will my refund be sent", the correct document *was* among the 5 candidates and the reranker still preferred another refund document. Rerankers raise precision; they aren't oracles.
- **The ceiling:** a reranker only reorders what retrieval handed it, so if the right chunk isn't in the top-k it can never come back (`rg08`). **Recall@k of the first stage is the ceiling**, and the assert checks it. Measure recall@k before spending latency on reranking.
- **Cost:** reranking k documents costs k forward passes per query, which is several times the retrieval cost even at k = 5. Production systems typically rerank 20–100 candidates within a latency budget. The ms-marco reranker was trained on web-search queries, so it is strongest on question-style inputs; evaluate on your own traffic.
- **Scores are logits and are only comparable within one query.** Don't use one global threshold across queries without calibrating it.

| | Bi-encoder (embedding model) | Cross-encoder (reranker) |
|---|---|---|
| Input | one text at a time | (query, document) together |
| Output | a vector per text | one relevance score per pair |
| Precompute documents? | **yes**: embed once, store in a vector DB | **no**: must run at query time |
| Cost per query | 1 forward pass + a vector lookup | k forward passes (one per candidate) |
| Accuracy | good at topic, weaker at fine distinctions | higher: full cross-attention between query and doc |
| Scales to | millions to billions of documents (with ANN) | tens to hundreds of candidates |
| Trained with | contrastive loss, in-batch negatives | relevance classification on (query, passage) labels |
| Role | first stage: **recall** | second stage: **precision** |

## 7. Fine-tuning an embedding model (MultipleNegativesRankingLoss)

When your domain uses words the base model never saw (internal acronyms, product codes, legal terms), a few thousand **(query, relevant passage)** pairs can improve retrieval a lot (`rg38`).

**MultipleNegativesRankingLoss (MNRL)** needs only positive pairs. In a batch of *B* pairs, each query's own passage is its positive and the other *B − 1* passages act as **in-batch negatives**. The loss is cross-entropy over the similarity matrix, where the "correct class" for query *i* is passage *i*. Bigger batches mean more negatives and usually a better model, and adding **hard negatives** (passages that look relevant but aren't) as a third column helps most. Use `BatchSamplers.NO_DUPLICATES` so that a batch never contains the same passage twice, which would count a positive as a negative.

A tiny runnable version: 8 pairs of company jargon, 3 epochs, a few seconds on CPU. It shows the mechanics and the effect on *these* pairs. A real run needs thousands of pairs and a held-out recall@k evaluation.

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from transformers.trainer_callback import PrinterCallback
import contextlib, io

jargon = [("How much PTO do I get?", "Employees receive 25 days of paid time off per year."),
          ("What is the RMA process?", "To return a faulty device, request a return merchandise authorization number."),
          ("Who approves my OOO?", "Your line manager must approve out-of-office requests in the HR portal."),
          ("Where do I file a TPS report?", "Weekly testing status reports are uploaded to the QA share."),
          ("How do I get a new SSO key?", "Single sign-on hardware keys are issued by the IT service desk."),
          ("What is our SLA for P1 tickets?", "Priority-one incidents must be acknowledged within 15 minutes."),
          ("When is the OKR review?", "Objectives and key results are reviewed every quarter."),
          ("Can I expense a WFH desk?", "Home-office furniture can be reimbursed up to 300 euros.")]
queries, passages = [q for q, _ in jargon], [p for _, p in jargon]

def retrieval_quality(m):
    S = m.encode(queries, normalize_embeddings=True) @ m.encode(passages, normalize_embeddings=True).T
    return float(np.mean(np.diag(S))), float(np.mean(S.argmax(axis=1) == np.arange(len(queries))))

ft_model = SentenceTransformer(EMB, device="cpu")           # a fresh copy; `model` stays the base model
before = retrieval_quality(ft_model)
args = SentenceTransformerTrainingArguments(
    output_dir=tempfile.mkdtemp(), num_train_epochs=3, per_device_train_batch_size=4, learning_rate=5e-5,
    batch_sampler=BatchSamplers.NO_DUPLICATES, logging_steps=2, save_strategy="no", report_to="none",
    use_cpu=True, seed=0, disable_tqdm=True)
t0 = time.perf_counter()
with contextlib.redirect_stderr(io.StringIO()):            # hide a model-card progress bar
    trainer = SentenceTransformerTrainer(model=ft_model, args=args,
                                         train_dataset=Dataset.from_dict({"anchor": queries, "positive": passages}),
                                         loss=MultipleNegativesRankingLoss(ft_model))
    trainer.remove_callback(PrinterCallback)               # we print the losses ourselves below
    trainer.train()
after = retrieval_quality(ft_model)
losses = [r["loss"] for r in trainer.state.log_history if "loss" in r]
print(f"\ntrained in {time.perf_counter() - t0:.1f}s; loss per logging step: {np.round(losses, 3)}")
print(f"before: mean positive cosine {before[0]:.3f}, top-1 {before[1]:.2f}")
print(f"after : mean positive cosine {after[0]:.3f}, top-1 {after[1]:.2f}")
assert losses[-1] < losses[0] and after[1] == 1.0 > before[1]

Before training, the base model fails on some of the jargon queries (an acronym such as "TPS" or "OOO" means nothing to it). After three epochs on the pairs themselves, every query finds its passage. That is **training-set** performance, so it proves the loss works, not that the model generalises. In a real project: mine pairs from logs or tickets, add hard negatives, hold out 10–20 % of queries, report recall@k and MRR against the base model, and budget for **re-embedding the whole corpus** (a new model means a new vector space; old and new vectors can't be mixed).

## Try it yourself

**1. Dot product vs cosine.** Encode two sentences *without* `normalize_embeddings` and check the vector norms: is this model's raw output already normalised? Then scale one vector by 3 (as a model that encodes "confidence" in the length might) and show that the dot product changes but the cosine does not.

In [ ]:
# Solution — try it yourself first, then run this
raw = model.encode(["A man is eating food.", "Someone is having a meal."], normalize_embeddings=False)
print(f"norms of raw vectors: {np.linalg.norm(raw, axis=1).round(3)}  (this model ends with a Normalize module)")

def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

long_vec = 3 * raw[1]
print(f"dot    : {raw[0] @ raw[1]:.4f} -> {raw[0] @ long_vec:.4f} after scaling")
print(f"cosine : {cosine(raw[0], raw[1]):.4f} -> {cosine(raw[0], long_vec):.4f} after scaling")
assert np.isclose(cosine(raw[0], raw[1]), cosine(raw[0], long_vec)) and not np.isclose(raw[0] @ raw[1], raw[0] @ long_vec)
# With a model that does not normalise, raw dot product also rewards vector length:
# normalise both sides (then dot == cosine), unless the model was trained for dot product.

**2. How deep should you rerank?** For each question in `eval_set`, record the bi-encoder rank of the correct document (retrieve all 16). What `top_k` would the reranker need to see every correct answer?

In [ ]:
# Solution — try it yourself first, then run this
ranks = []
for q, gold in eval_set:
    order = [h["corpus_id"] for h in util.semantic_search(
        model.encode(q, normalize_embeddings=True, convert_to_tensor=True), doc_emb, top_k=len(docs))[0]]
    ranks.append(order.index(gold) + 1)
print("rank of the correct doc per question:", ranks)
print("recall@k:", {k: float(np.mean(np.array(ranks) <= k)) for k in [1, 2, 3, 5]})
print("smallest top_k that contains every answer:", max(ranks))

**3. Semantic dedup.** Using `util.paraphrase_mining` (or a similarity matrix), find the most similar pair among the 24 clustering sentences plus two near-duplicates you add. Are the near-duplicates the top pair?

In [ ]:
# Solution — try it yourself first, then run this
extra = texts + ["Stock prices dropped after the earnings report.", "Preheat the oven before you bake the cake."]
pairs = util.paraphrase_mining(model, extra, top_k=3)
for score, i, j in pairs[:3]:
    print(f"{score:.3f}  {extra[i]!r}  <->  {extra[j]!r}")
assert {pairs[0][1], pairs[0][2]} & {len(extra) - 1, len(extra) - 2}

## Say it in an interview

- **Bi-encoder vs cross-encoder:** "An embedding model encodes query and document independently, so I can precompute document vectors and search millions with ANN. A cross-encoder reads the pair together, which is more accurate but needs one forward pass per pair. So: retrieve the top 50–100 with the bi-encoder, rerank with the cross-encoder, and pass the top 5 to the LLM."
- **The reranker ceiling:** it can't fix recall. If the right chunk isn't in the candidates, reranking never sees it, so measure recall@k of stage one first.
- **Normalisation:** unit vectors make dot product equal cosine. Normalise queries and documents the same way. Use raw dot product only if the model was trained for it.
- **Operational facts:** embedding is one forward pass and very batchable (throughput climbs with batch size); `max_seq_length` truncates silently (256 tokens here), hence chunking; changing the embedding model means re-embedding everything.
- **Choosing a model (`tl18`):** retrieval benchmarks (MTEB) as a first filter, then *your* eval set; dimension (storage and speed), max length, language coverage, licence, and hosted vs self-hosted.
- **Fine-tuning:** (query, positive) pairs with MNRL and in-batch negatives, plus hard negatives; evaluate held-out recall@k and MRR against the base model; plan the re-index.

## Next

- Build the retrieval stack: [vector search with FAISS/HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb) · [hybrid search and RRF](../09_rag/04_hybrid_search_and_rrf.ipynb) · [reranking](../09_rag/05_reranking.ipynb) · [RAG from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb)
- Evaluate it: [RAG evaluation: retrieval and faithfulness](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb)
- In an interview: [semantic search service](../17_interview_coding_rounds/04_semantic_search_service.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html), [ai_system_design_concepts](../../ai_system_design_concepts/index.html), [interview bank](../../ai_interview_prep/index.html)